# SAR-to-EO Image Translation — GalaxEye Assignment
**Model:** Pix2Pix U-Net + PatchGAN

---
## ⚡ Every Session Checklist
1. Settings → Accelerator → **GPU T4 x2** (pick the best available)
2. Add dataset: `sentinel12-image-pairs-segregated-by-terrain`
3. Run **Cells 1 → 2 → 3 → 4** (setup), then the relevant training cell
4. **IMMEDIATELY after training finishes → run the DOWNLOAD cell below it**
5. Download files from the Output panel on the right

---
## 🔀 Two-Account Parallel Plan
Run both accounts simultaneously to save time!

| Account | Run these cells | Time |
|---|---|---|
| **Account 1** (main) | 1→2→3→4 → **7d** → DOWNLOAD D | ~7.5 hrs |
| **Account 2** (second) | 1→2→3→4 → **7b** → **7c** → DOWNLOAD BC | ~9.3 hrs |
| **Eval session** (any account) | 1→2→3→4 → RESTORE → **Cell 8** → DOWNLOAD RESULTS | ~1.5 hrs |

**Config A** is already done (best_l1_only.pth downloaded to laptop ✅)

---
## ⚠️ Download Rule
**Never leave files undownloaded when the session is about to expire.**
Files in `/kaggle/working/` are wiped when the session ends.

In [ ]:
# ================================================================
# CELL 1: Environment check
# ================================================================
import subprocess, torch, os
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'GPU {i}  : {torch.cuda.get_device_name(i)}')
        print(f'VRAM {i} : {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB')

In [ ]:
# ================================================================
# CELL 2: Install packages
# ================================================================
!pip install lpips pytorch-fid scikit-image pyyaml -q
print('Packages ready.')

In [ ]:
# ================================================================
# CELL 3: Clone repo
# NOTE: Reset cwd FIRST to avoid 'getcwd: no such directory' crash
# ================================================================
import os, shutil, subprocess

os.chdir('/kaggle/working')   # CRITICAL: reset cwd before anything else

REPO_DIR = '/kaggle/working/sar2eo'
REPO_URL = 'https://github.com/Trafalgar-2006/sar2eo.git'

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

result = subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], capture_output=True, text=True)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise RuntimeError('Git clone failed')

os.chdir(REPO_DIR)
print('Repo cloned. Working dir:', os.getcwd())
print('Contents:', os.listdir('.'))

In [ ]:
# ================================================================
# CELL 4: Find dataset + configure paths
# ================================================================
import yaml, os
from pathlib import Path

# --- Find Sentinel dataset ---
TERRAIN_NAMES = {'agri', 'urban', 'grassland', 'barrenland'}

def _has_terrain_dirs(path):
    try:
        subdirs = {d.name.lower() for d in Path(path).iterdir() if d.is_dir()}
        return bool(subdirs & TERRAIN_NAMES)
    except:
        return False

DATASET_ROOT = None
for candidate in sorted(Path('/kaggle/input').rglob('*')):
    if candidate.is_dir() and _has_terrain_dirs(candidate):
        DATASET_ROOT = str(candidate)
        break

assert DATASET_ROOT, (
    'Sentinel dataset not found! '
    'Add: sentinel12-image-pairs-segregated-by-terrain\n'
    'Inputs found: ' + str(list(Path('/kaggle/input').iterdir()))
)
print(f'Dataset  : {DATASET_ROOT}')
print(f'Contents : {[d.name for d in Path(DATASET_ROOT).iterdir() if d.is_dir()]}')

# --- Configure config.yaml ---
os.chdir('/kaggle/working/sar2eo')
with open('config.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['data']['dataset_type']    = 'kaggle'
cfg['data']['kaggle_root']     = DATASET_ROOT
cfg['data']['train_terrain']   = ['agri', 'barrenland', 'grassland']
cfg['data']['val_terrain']     = ['urban']
cfg['data']['test_terrain']    = ['urban']
cfg['data']['subset_size']     = None
cfg['data']['num_workers']     = 2
cfg['training']['batch_size']  = 16
cfg['training']['epochs']      = 100
cfg['paths']['checkpoint_dir'] = '/kaggle/working/sar2eo/checkpoints'
cfg['paths']['output_dir']     = '/kaggle/working/sar2eo/outputs'

with open('config.yaml', 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print('\nConfig ready:')
print(f'  dataset     : {DATASET_ROOT}')
print(f'  checkpoints : /kaggle/working/sar2eo/checkpoints')
print(f'  batch_size  : {cfg["training"]["batch_size"]}')
print(f'  epochs      : {cfg["training"]["epochs"]}')

---
## 🔴 ACCOUNT 1 ONLY: Config D — Full model (MAIN SUBMISSION) | ~7.5 hrs
Run **7d**, then immediately run **DOWNLOAD D** below it.

In [ ]:
# ================================================================
# CELL 7d: Config D — Full model (L1 + Adv + FFT + VGG)
# ~7.5 hrs (100 epochs) | Auto-resumes from checkpoint if exists
# ACCOUNT 1 ONLY
# ================================================================
import subprocess, os
os.chdir('/kaggle/working/sar2eo')
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'

print('=' * 55)
print('Config D: Full model (MAIN SUBMISSION)  |  ~7.5 hrs')
print('100 epochs | L1 + Adversarial + FFT + VGG Perceptual')
print('=' * 55)

r = subprocess.run(
    ['python', '-u', 'train.py', '--config', 'config.yaml', '--ablation', 'full'],
    env=env
)
print('\nConfig D', 'DONE ✅' if r.returncode == 0 else 'FAILED ❌')
print('>>> Run DOWNLOAD D cell below NOW <<<')

In [ ]:
# ================================================================
# DOWNLOAD D — Run immediately after Config D finishes
# Downloads: best.pth + loss curve + losses CSV
# Then download from the Output panel on the right →
# ================================================================
import shutil, os, glob
os.chdir('/kaggle/working/sar2eo')

ABLATION = 'full'
CKPT_DIR = f'checkpoints/{ABLATION}'

# Verify checkpoint exists
best_pth = f'{CKPT_DIR}/best.pth'
assert os.path.exists(best_pth), f'ERROR: {best_pth} not found! Did training complete?'

# Copy to /kaggle/working/ root (Output panel shows files here)
shutil.copy(best_pth, '/kaggle/working/best_full.pth')

curve = f'outputs/loss_curve_{ABLATION}.png'
losses = f'outputs/losses_{ABLATION}.csv'
if os.path.exists(curve):  shutil.copy(curve,  f'/kaggle/working/loss_curve_full.png')
if os.path.exists(losses): shutil.copy(losses, f'/kaggle/working/losses_full.csv')

# Also copy sample images
sample_src = f'outputs/samples/{ABLATION}'
if os.path.exists(sample_src):
    shutil.copytree(sample_src, '/kaggle/working/samples_full', dirs_exist_ok=True)

print('✅ Files ready to download from Output panel:')
for f in sorted(glob.glob('/kaggle/working/best_full*') +
                glob.glob('/kaggle/working/loss_curve_full*') +
                glob.glob('/kaggle/working/losses_full*')):
    print(f'  {os.path.basename(f)}: {os.path.getsize(f)/1e6:.1f} MB')

print('\n>>> DOWNLOAD best_full.pth + loss_curve_full.png + losses_full.csv NOW <<<')

---
## 🔵 ACCOUNT 2 ONLY: Config B + C | ~9.3 hrs total
Run **7b** (waits ~4.5 hrs), then **7c** (waits ~4.8 hrs), then **DOWNLOAD BC**.

In [ ]:
# ================================================================
# CELL 7b: Config B — L1 + Adversarial
# ~4.5 hrs (100 epochs) | Auto-resumes from checkpoint if exists
# ACCOUNT 2 ONLY
# ================================================================
import subprocess, os
os.chdir('/kaggle/working/sar2eo')
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'

print('=' * 55)
print('Config B: L1 + Adversarial (Pix2Pix)  |  ~4.5 hrs')
print('=' * 55)

r = subprocess.run(
    ['python', '-u', 'train.py', '--config', 'config.yaml', '--ablation', 'l1_adv'],
    env=env
)
print('\nConfig B', 'DONE ✅' if r.returncode == 0 else 'FAILED ❌')
print('>>> Run Cell 7c immediately below <<<')

In [ ]:
# ================================================================
# CELL 7c: Config C — L1 + Adversarial + FFT
# ~4.8 hrs (100 epochs) | Auto-resumes from checkpoint if exists
# ACCOUNT 2 ONLY
# ================================================================
import subprocess, os
os.chdir('/kaggle/working/sar2eo')
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'

print('=' * 55)
print('Config C: L1 + Adversarial + FFT  |  ~4.8 hrs')
print('=' * 55)

r = subprocess.run(
    ['python', '-u', 'train.py', '--config', 'config.yaml', '--ablation', 'l1_adv_fft'],
    env=env
)
print('\nConfig C', 'DONE ✅' if r.returncode == 0 else 'FAILED ❌')
print('>>> Run DOWNLOAD BC cell below NOW <<<')

In [ ]:
# ================================================================
# DOWNLOAD BC — Run after BOTH Config B and C finish
# Copies B and C artifacts to Output panel for download
# ================================================================
import shutil, os, glob
os.chdir('/kaggle/working/sar2eo')

for ablation, name in [('l1_adv', 'B'), ('l1_adv_fft', 'C')]:
    best_pth = f'checkpoints/{ablation}/best.pth'
    if not os.path.exists(best_pth):
        print(f'WARNING: {best_pth} missing — Config {name} may not have finished')
        continue

    shutil.copy(best_pth, f'/kaggle/working/best_{ablation}.pth')

    curve  = f'outputs/loss_curve_{ablation}.png'
    losses = f'outputs/losses_{ablation}.csv'
    if os.path.exists(curve):  shutil.copy(curve,  f'/kaggle/working/loss_curve_{ablation}.png')
    if os.path.exists(losses): shutil.copy(losses, f'/kaggle/working/losses_{ablation}.csv')

    sample_src = f'outputs/samples/{ablation}'
    if os.path.exists(sample_src):
        shutil.copytree(sample_src, f'/kaggle/working/samples_{ablation}', dirs_exist_ok=True)

    print(f'✅ Config {name} ({ablation}) ready')

print('\n✅ All files ready. Download from Output panel:')
for f in sorted(glob.glob('/kaggle/working/best_l1*.pth') +
                glob.glob('/kaggle/working/loss_curve_l1*.png') +
                glob.glob('/kaggle/working/losses_l1*.csv')):
    print(f'  {os.path.basename(f)}: {os.path.getsize(f)/1e6:.1f} MB')

print('\n>>> DOWNLOAD all 6 files (2x .pth + 2x .png + 2x .csv) NOW <<<')

---
## 🟢 EVAL SESSION (any account) — Run after all 4 configs trained
**Prerequisites:** You have downloaded these 4 files to your laptop:
- `best_l1_only.pth` (Config A — from Session 1)
- `best_l1_adv.pth` (Config B)
- `best_l1_adv_fft.pth` (Config C)
- `best_full.pth` (Config D)

**Step 1:** Upload all 4 files using the **Upload** button in the Input panel (right sidebar)

**Step 2:** Run: Cells 1 → 2 → 3 → 4 → RESTORE → Cell 8 → DOWNLOAD RESULTS

In [ ]:
# ================================================================
# RESTORE — Place uploaded checkpoints into correct directories
# Run AFTER uploading all 4 best.pth files via Input → Upload
# ================================================================
import os, shutil, glob
os.chdir('/kaggle/working/sar2eo')

# Map: filename → ablation folder name
CHECKPOINT_MAP = {
    'best_l1_only.pth':    'l1_only',
    'best_l1_adv.pth':     'l1_adv',
    'best_l1_adv_fft.pth': 'l1_adv_fft',
    'best_full.pth':       'full',
}

# Search for uploaded files in /kaggle/input/ and /kaggle/working/
search_dirs = ['/kaggle/input/', '/kaggle/working/']
restored = 0

for filename, ablation in CHECKPOINT_MAP.items():
    found = None
    for search_dir in search_dirs:
        matches = list(glob.glob(f'{search_dir}/**/{filename}', recursive=True)) + \
                  [f'{search_dir}{filename}'] if os.path.exists(f'{search_dir}{filename}') else \
                  list(glob.glob(f'{search_dir}/**/{filename}', recursive=True))
        if matches:
            found = matches[0]
            break

    if found:
        dest_dir = f'/kaggle/working/sar2eo/checkpoints/{ablation}'
        os.makedirs(dest_dir, exist_ok=True)
        dest = f'{dest_dir}/best.pth'
        shutil.copy(found, dest)
        sz = os.path.getsize(dest) / 1e6
        print(f'✅ {ablation}/best.pth restored ({sz:.0f} MB) from {found}')
        restored += 1
    else:
        print(f'❌ {filename} NOT FOUND — upload it via Input → Upload button')

print(f'\n{restored}/4 checkpoints restored.')
if restored == 4:
    print('✅ All 4 ready. Run Cell 8 (Evaluate all configs).')
else:
    print('⚠️ Upload missing files before running Cell 8.')

In [ ]:
# ================================================================
# CELL 8: Evaluate all configs
# Requires: all 4 best.pth files in checkpoints/{ablation}/best.pth
# Run RESTORE cell first if starting a new session
# ================================================================
import subprocess, os
os.chdir('/kaggle/working/sar2eo')
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'

ABLATIONS = ['l1_only', 'l1_adv', 'l1_adv_fft', 'full']
results = {}

for ablation in ABLATIONS:
    ckpt = f'/kaggle/working/sar2eo/checkpoints/{ablation}/best.pth'
    if not os.path.exists(ckpt):
        print(f'  Skip {ablation} — no best.pth found')
        continue
    print(f'\n{"="*50}')
    print(f'Evaluating: {ablation}')
    print(f'{"="*50}')
    r = subprocess.run([
        'python', '-u', 'eval.py',
        '--config', 'config.yaml',
        '--weights', ckpt,
        '--split', 'test'
    ], env=env)
    results[ablation] = 'DONE' if r.returncode == 0 else 'FAILED'

print('\n' + '='*50)
print('EVALUATION SUMMARY:')
for ablation, status in results.items():
    print(f'  {ablation}: {status}')
print('\n>>> Run DOWNLOAD RESULTS cell below <<<')

In [ ]:
# ================================================================
# DOWNLOAD RESULTS — Run after Cell 8 (Eval) finishes
# Copies all metrics CSVs and visualizations to Output panel
# ================================================================
import shutil, os, glob
os.chdir('/kaggle/working/sar2eo')

# Copy eval metrics for all configs
os.makedirs('/kaggle/working/eval_results', exist_ok=True)

for ablation in ['l1_only', 'l1_adv', 'l1_adv_fft', 'full']:
    for pattern in [f'outputs/metrics_{ablation}*.csv',
                    f'outputs/results_{ablation}*.csv',
                    f'outputs/*{ablation}*metrics*',
                    f'outputs/eval_{ablation}*']:
        for src in glob.glob(pattern):
            dst = f'/kaggle/working/eval_results/{os.path.basename(src)}'
            shutil.copy(src, dst)
            print(f'Copied: {os.path.basename(src)}')

# Also copy all CSVs and visualizations from outputs/
for f in glob.glob('outputs/**/*.csv', recursive=True) + \
         glob.glob('outputs/**/*.png', recursive=True):
    dst = f'/kaggle/working/eval_results/{os.path.basename(f)}'
    if not os.path.exists(dst):
        shutil.copy(f, dst)

print('\n✅ Results ready in eval_results/ folder:')
files = sorted(glob.glob('/kaggle/working/eval_results/*'))
for f in files:
    print(f'  {os.path.basename(f)}: {os.path.getsize(f)/1e6:.2f} MB')

if not files:
    print('WARNING: No results found. Check that eval.py ran successfully.')